# Two-island model

Formålet med denne notebook er at implementere en klassisk two-island model, hvor populationen består af to subpopulationer med migration mellem dem.

Denne model er central i populationsgenetik og bruges til at beskrive:
- populationsstruktur
- gene flow
- divergens mellem populationer

Den fungerer også som basis for:
- IM modeller
- ghost population modeller

In [ ]:
# ALWAYS import phasic first
from phasic import Graph, StateIndexer, Property

import numpy as np
import matplotlib.pyplot as plt

from vscodenb import set_vscode_theme
set_vscode_theme()

np.random.seed(42)

## Modelbeskrivelse

Jeg har to populationer:
- pop1
- pop2

Lineager kan:
1. Coalescere inden for samme population
2. Migrere mellem populationer

Parametre:
- θ₁, θ₂: coalescent rates
- m: migrationsrate (symmetrisk)

Transition rates:

Coalescence:
$$
\lambda_{coal} = \frac{x_i (x_j - \delta_{ij})}{1 + \delta_{ij}}
$$

Migration:
$$
\lambda_{mig} = m
$$

Denne model antager konstant populationsstørrelse og konstant migration.

## Stateindexer

In [ ]:
nr_samples = 2

indexer = StateIndexer(
    descendants=[
        Property('loc1', max_value=nr_samples),
        Property('loc2', max_value=nr_samples)
    ]
)

## Initial state

In [ ]:
initial = [0] * indexer.state_length

initial[indexer.props_to_index(loc1=1, loc2=1)] = nr_samples

Starter med en lineage i hver population

## Two-island model 

In [ ]:
def two_island_model(state, indexer=None):
    transitions = []
    
    if state.sum() <= 1:
        return transitions
    
    for i in range(indexer.state_length):
        if state[i] == 0:
            continue
        
        pi = indexer.index_to_props(i)
        
        for j in range(i, indexer.state_length):
            if state[j] == 0:
                continue
            
            pj = indexer.index_to_props(j)
            
            same = int(i == j)
            
            # COALESCENCE 
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue
            
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            
            loc1 = pi.descendants.loc1 + pj.descendants.loc1
            loc2 = pi.descendants.loc2 + pj.descendants.loc2
            
            if loc1 <= nr_samples and loc2 <= nr_samples:
                child[indexer.props_to_index(loc1=loc1, loc2=loc2)] += 1
                
                transitions.append([
                    child,
                    [state[i]*(state[j]-same)/(1+same), 0]
                ])
            
            # MIGRATION SPLIT 
            if state[i] > 0 and pi.descendants.loc1 > 0 and pi.descendants.loc2 > 0:
                
                child = state.copy()
                child[i] -= 1
                
                child[indexer.props_to_index(loc1=pi.descendants.loc1, loc2=0)] += 1
                child[indexer.props_to_index(loc1=0, loc2=pi.descendants.loc2)] += 1
                
                transitions.append([child, [0, 1]])
    
    return transitions

In [ ]:
graph = Graph(
    two_island_model,
    ipv=initial,
    indexer=indexer
)

graph.plot()

In [ ]:
theta = 1.0
migration = 0.5

graph.update_weights([theta, migration])

In [ ]:
print("Expectation:", graph.expectation())
print("Variance:", graph.variance())

## Simulation

In [ ]:
samples = graph.sample(2000)

plt.hist(samples, bins=50, density=True)
plt.title("Two-island coalescent times")
plt.show()

In [ ]:
migration_rates = [0.1, 0.5, 1.0, 2.0]

for m in migration_rates:
    graph.update_weights([theta, m])
    samples = graph.sample(2000)
    
    plt.hist(samples, bins=40, density=True, alpha=0.4, label=f"m={m}")

plt.legend()
plt.title("Effect of migration")
plt.show()

Migration påvirker coalescent tider:

- Lav migration:
  → populationer er adskilte
  → længere coalescent tider

- Høj migration:
  → populationer blandes
  → hurtigere coalescence

Dette giver et direkte signal i genetiske data.

In [ ]:
graph = Graph(
    two_island_model,
    ipv=initial,
    indexer=indexer,
    cache_graph=True
)